In [23]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader, Subset
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from torch_sparse import SparseTensor

from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [24]:
def setup_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [25]:
import numpy as np

feature_path = "/home/snu/Downloads/liver_dino_features.npy"

label_path = "/home/snu/Downloads/liver_dino_labels.npy"

features_data = np.load(
    feature_path
).astype(np.float32)

y_labels = np.load(
    label_path
).astype(np.int64)

print("Feature shape:", features_data.shape)

print("Label shape:", y_labels.shape)

print("\nClass distribution:")

unique, counts = np.unique(
    y_labels,
    return_counts=True
)

for cls, cnt in zip(unique, counts):

    print(f"Class {cls}: {cnt}")

Feature shape: (635, 768)
Label shape: (635,)

Class distribution:
Class 0: 200
Class 1: 435


In [26]:
def create_adj(features, alpha=0.7):
    f = features / np.linalg.norm(features, axis=1, keepdims=True)
    W = np.dot(f, f.T)
    W = (W >= alpha).astype(np.float32)
    return W

N = features_data.shape[0]
F_dim = features_data.shape[1]
x = torch.from_numpy(features_data).to(device)
y = torch.from_numpy(y_labels).to(device)
W = create_adj(features_data, alpha=0.7)

rows, cols = np.nonzero(W)
edge_index = torch.tensor([rows, cols], dtype=torch.long)
edge_index = to_undirected(edge_index).to(device)

adj = SparseTensor(
    row=edge_index[0],
    col=edge_index[1],
    sparse_sizes=(N, N)
).fill_value(1.).to(device)

print("Edges:", adj.nnz())

Edges: 337769


In [27]:
def get_sim(batch, adj, wt=50, wl=2):
    batch_size = batch.shape[0]
    batch_repeat = batch.repeat(wt)
    rw = adj.random_walk(batch_repeat, wl)[:, 1:]
    rw = rw.t().reshape(-1, batch_size).t()

    row, col, val = [], [], []
    for i in range(batch_size):
        nodes, counts = torch.unique(rw[i], return_counts=True)
        row += [batch[i].item()] * nodes.shape[0]
        col += nodes.tolist()
        val += counts.tolist()

    adj_rw = SparseTensor(
        row=torch.tensor(row),
        col=torch.tensor(col),
        value=torch.tensor(val),
        sparse_sizes=(batch_size, batch_size)
    )
    return adj_rw.set_diag(0.)


In [28]:
def get_mask(adj):
    mean = adj.mean(dim=1)
    mask = (adj.storage.value() -
            mean[adj.storage.row()]) > -1e-10

    return SparseTensor(
        row=adj.storage.row()[mask],
        col=adj.storage.col()[mask],
        value=adj.storage.value()[mask],
        sparse_sizes=adj.sizes()
    )

def scale(z):
    zmin = z.min(dim=1, keepdim=True)[0]
    zmax = z.max(dim=1, keepdim=True)[0]
    return (z - zmin) / (zmax - zmin + 1e-12)

In [29]:
class MAGILoss(nn.Module):
    def __init__(self, tau=0.3):
        super().__init__()
        self.tau = tau

    def forward(self, z, mask):
        sim = torch.mm(z, z.t()) / self.tau
        sim = sim - sim.max(dim=1, keepdim=True)[0].detach()

        logits_mask = torch.ones_like(sim) - torch.eye(z.size(0), device=z.device)
        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True))

        row, col = mask.storage.row(), mask.storage.col()
        return -log_prob[row, col].mean()

In [30]:
class Encoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256):
        super().__init__()
        self.conv = GCNConv(in_dim, hidden_dim)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        return F.leaky_relu(x, 0.2)

In [31]:
from sklearn.metrics import normalized_mutual_info_score

n_runs  = 10
epochs  = 3000

acc_list, prec_list, rec_list, f1_list, nmi_list = [], [], [], [], []

for run in range(n_runs):
    print(f"\n========== Run {run+1}/{n_runs} ==========")

    setup_seed(42 + run)

    encoder   = Encoder(F_dim, 256).to(device)
    criterion = MAGILoss(tau=0.3)
    optimizer = torch.optim.Adam(
        encoder.parameters(), lr=5e-4, weight_decay=1e-3
    )

    batch  = torch.arange(N, device=device)
    adj_rw = get_sim(batch, adj)
    mask   = get_mask(adj_rw)

    for epoch in range(epochs):
        encoder.train()
        optimizer.zero_grad()

        z    = encoder(x, edge_index)
        z    = scale(z)
        z    = F.normalize(z, dim=1)
        loss = criterion(z, mask)
        loss.backward()
        optimizer.step()

        if epoch % 500 == 0:
            print(f"Run {run+1} | Epoch {epoch} | Loss {loss.item():.4f}")

    # -------------------- Evaluation (final model only) --------------------
    encoder.eval()
    with torch.no_grad():
        z = encoder(x, edge_index)
        z = scale(z)
        z = F.normalize(z, dim=1).cpu().numpy()

    kmeans = KMeans(n_clusters=2, n_init=20, random_state=run)
    y_pred = kmeans.fit_predict(z)

    # Convert y to numpy array on CPU for sklearn metrics
    y_np = y.cpu().numpy()

    acc1 = accuracy_score(y_np, y_pred)
    acc2 = accuracy_score(y_np, 1 - y_pred)
    if acc2 > acc1:
        y_pred = 1 - y_pred

    acc  = accuracy_score(y_np, y_pred)
    prec = precision_score(y_np, y_pred, zero_division=0)
    rec  = recall_score(y_np, y_pred,    zero_division=0)
    f1   = f1_score(y_np, y_pred,        zero_division=0)

    # ── NMI: one value per run, final model only ──
    nmi  = normalized_mutual_info_score(y_np, y_pred, average_method='arithmetic')

    acc_list.append(acc)
    prec_list.append(prec)
    rec_list.append(rec)
    f1_list.append(f1)
    nmi_list.append(nmi)

    print(
        f"Run {run+1} → "
        f"ACC: {acc:.4f} | "
        f"PREC: {prec:.4f} | "
        f"REC: {rec:.4f} | "
        f"F1: {f1:.4f} | "
        f"NMI: {nmi:.4f}"
    )

print("\n===== MAGI + K-Means (10 Runs) =====")
print(f"ACC : {np.mean(acc_list):.4f} \u00b1 {np.std(acc_list):.4f}")
print(f"PREC: {np.mean(prec_list):.4f} \u00b1 {np.std(prec_list):.4f}")
print(f"REC : {np.mean(rec_list):.4f} \u00b1 {np.std(rec_list):.4f}")
print(f"F1  : {np.mean(f1_list):.4f} \u00b1 {np.std(f1_list):.4f}")
print(f"NMI : {np.mean(nmi_list):.4f} \u00b1 {np.std(nmi_list):.4f}")


========== Run 1/10 ==========
Run 1 | Epoch 0 | Loss 6.4502
Run 1 | Epoch 500 | Loss 6.4170
Run 1 | Epoch 1000 | Loss 6.4171
Run 1 | Epoch 1500 | Loss 6.4167
Run 1 | Epoch 2000 | Loss 6.4173
Run 1 | Epoch 2500 | Loss 6.4175
Run 1 → ACC: 0.5071 | PREC: 0.8144 | REC: 0.3632 | F1: 0.5024 | NMI: 0.0293

========== Run 2/10 ==========
Run 2 | Epoch 0 | Loss 6.4499
Run 2 | Epoch 500 | Loss 6.4198
Run 2 | Epoch 1000 | Loss 6.4179
Run 2 | Epoch 1500 | Loss 6.4184
Run 2 | Epoch 2000 | Loss 6.4179
Run 2 | Epoch 2500 | Loss 6.4177
Run 2 → ACC: 0.6913 | PREC: 0.8153 | REC: 0.7103 | F1: 0.7592 | NMI: 0.0895

========== Run 3/10 ==========
Run 3 | Epoch 0 | Loss 6.4501
Run 3 | Epoch 500 | Loss 6.4220
Run 3 | Epoch 1000 | Loss 6.4200
Run 3 | Epoch 1500 | Loss 6.4199
Run 3 | Epoch 2000 | Loss 6.4217
Run 3 | Epoch 2500 | Loss 6.4195
Run 3 → ACC: 0.5638 | PREC: 0.8292 | REC: 0.4575 | F1: 0.5896 | NMI: 0.0481

========== Run 4/10 ==========
Run 4 | Epoch 0 | Loss 6.4499
Run 4 | Epoch 500 | Loss 6.4188
